# Customer Feedback Analysis and Automated Response

## Objective

The objective of this project is to analyze customer reviews,
identify critical negative feedback using rule-based filtering,
find common complaint keywords, and generate personalized
apology emails using Generative AI.

In [7]:
import pandas as pd
import re
from collections import Counter

In [10]:
df = pd.read_csv(r"C:\Users\Abdul Razak\Downloads\Dataset.csv\flipkart review dataset.csv", encoding="latin1", low_memory=False)

df.head(30)

,Product_name,Price,Rate,Review,Summary
0,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Simply awesome,it's really worth every single penny. it works...
1,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",4,Worth the money . Desert Cooler live up to the...,I bought Crompton Ozone 75 Desert Air Cooler i...
2,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Worth every penny,GREAT packaging by seller. As this was the mos...
3,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Fabulous!,Delivery was delayed by two days except this e...
4,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",4,Nice product,A Good cooler by Crompton. The height of the c...
5,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Super!,"Can't go wrong with Crompton Brand, In Punjab ..."
6,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Terrific purchase,A great cooler in great price. Compact and Gre...
7,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Classy product,Its an awesome product and I am not making any...
8,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Perfect product!,Air throw is very good.????????Low noise and b...
9,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Its not a cooler its Ac cooler,Its been 6 days that i am using this cooler we...


In [11]:
df.shape

(363261, 5)

In [12]:
df.columns

Index(['Product_name', 'Price', 'Rate', 'Review', 'Summary'], dtype='object')

In [13]:
df.head()

,Product_name,Price,Rate,Review,Summary
0,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Simply awesome,it's really worth every single penny. it works...
1,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",4,Worth the money . Desert Cooler live up to the...,I bought Crompton Ozone 75 Desert Air Cooler i...
2,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Worth every penny,GREAT packaging by seller. As this was the mos...
3,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",5,Fabulous!,Delivery was delayed by two days except this e...
4,"Crompton 75 L Desert Air Cooler??(White, Tea...","??10,499",4,Nice product,A Good cooler by Crompton. The height of the c...


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363261 entries, 0 to 363260
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   Product_name  363246 non-null  object
 1   Price         363245 non-null  object
 2   Rate          363245 non-null  object
 3   Review        363239 non-null  object
 4   Summary       361239 non-null  object
dtypes: object(5)
memory usage: 13.9+ MB


In [15]:
df.isnull().sum()

Product_name      15
Price             16
Rate              16
Review            22
Summary         2022
dtype: int64

In [16]:
df["Rate"].value_counts(dropna=False)

Rate
5                                                              203832
4                                                               74252
1                                                               40077
3                                                               32098
2                                                               12980
NaN                                                                16
product_name                                                        3
Pigeon Favourite Electric Kettle????(1.5 L, Silver, Black)          1
Bajaj DX 2 L/W Dry Iron                                             1
Nova Plus Amaze NI 10 1100 W Dry Iron????(Grey & Turquoise)         1
Name: count, dtype: int64

In [17]:
df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")

In [18]:
df["Rate"].value_counts(dropna=False)

Rate
5.0    203832
4.0     74252
1.0     40077
3.0     32098
2.0     12980
NaN        22
Name: count, dtype: int64

In [20]:
df = df.dropna(subset=["Rate"])

In [21]:
df["Rate"].value_counts()

Rate
5.0    203832
4.0     74252
1.0     40077
3.0     32098
2.0     12980
Name: count, dtype: int64

In [22]:
df["Rate"] = df["Rate"].astype(int)

In [23]:
df.isnull().sum()

Product_name       0
Price              0
Rate               0
Review             6
Summary         2006
dtype: int64

In [24]:
df.duplicated().sum()

np.int64(53770)

In [36]:
df = df.drop_duplicates().copy()

In [37]:
df.duplicated().sum()

np.int64(0)

In [38]:
df["Review"] = df["Review"].fillna("")
df["Summary"] = df["Summary"].fillna("")

In [39]:
df[["Review", "Summary"]].isnull().sum()

Review     0
Summary    0
dtype: int64

In [41]:
df["Feedback"] = df["Review"] + " " + df["Summary"]

In [42]:
df[["Review", "Summary", "Feedback"]].head()

,Review,Summary,Feedback
0,Simply awesome,it's really worth every single penny. it works...,Simply awesome it's really worth every single ...
1,Worth the money . Desert Cooler live up to the...,I bought Crompton Ozone 75 Desert Air Cooler i...,Worth the money . Desert Cooler live up to the...
2,Worth every penny,GREAT packaging by seller. As this was the mos...,Worth every penny GREAT packaging by seller. A...
3,Fabulous!,Delivery was delayed by two days except this e...,Fabulous! Delivery was delayed by two days exc...
4,Nice product,A Good cooler by Crompton. The height of the c...,Nice product A Good cooler by Crompton. The he...


In [43]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [44]:
df["Clean_Feedback"] = df["Feedback"].apply(clean_text)

In [45]:
df[["Feedback", "Clean_Feedback"]].head()

,Feedback,Clean_Feedback
0,Simply awesome it's really worth every single ...,simply awesome its really worth every single p...
1,Worth the money . Desert Cooler live up to the...,worth the money desert cooler live up to the n...
2,Worth every penny GREAT packaging by seller. A...,worth every penny great packaging by seller as...
3,Fabulous! Delivery was delayed by two days exc...,fabulous delivery was delayed by two days exce...
4,Nice product A Good cooler by Crompton. The he...,nice product a good cooler by crompton the hei...


In [46]:
critical_reviews = df[df["Rate"] <= 2].copy()

In [47]:
print("Total critical reviews:", len(critical_reviews))

Total critical reviews: 50158


In [48]:
critical_reviews[["Product_name", "Rate", "Review", "Summary"]].head(10)

,Product_name,Rate,Review,Summary
12,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Don't waste your money,"Water pump is not throwing a lot of water, tha..."
19,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Unsatisfactory,Its vertical Swing quality is bad. When you se...
20,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Utterly Disappointed,Never purchase cooler from FlipkartMy 10500 wa...
60,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Unsatisfactory,Air throw is very disappointing.I thought it w...
65,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Did not meet expectations,It is old model and don't have outlet for drai...
68,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Don't waste your money,I baught this cooler looking at the number of ...
82,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Did not meet expectations,Not a good body. Delivery of this product is n...
155,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Not recommended at all,this cooler is wonderful I would say best dese...
156,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Product changed,I buy this product 55 dessert but total sent a...
157,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Item is good but Delhivery courier service worst.,Item is good but Delivery service of flipkart ...


In [49]:
critical_reviews["Rate"].value_counts()

Rate
1    37856
2    12302
Name: count, dtype: int64

In [50]:
critical_reviews["Severity"] = critical_reviews["Rate"].apply(
    lambda x: "Critical" if x <= 2 else "Not Critical"
)

In [52]:
critical_reviews[["Rate", "Severity"]].head(10)

,Rate,Severity
12,1,Critical
19,1,Critical
20,1,Critical
60,1,Critical
65,1,Critical
68,1,Critical
82,1,Critical
155,1,Critical
156,1,Critical
157,1,Critical


In [53]:
complaint_keywords = [
    "broken",
    "damaged",
    "damage",
    "late",
    "delay",
    "rude",
    "bad",
    "poor",
    "worst",
    "refund",
    "wrong",
    "missing",
    "defective",
    "quality",
    "disappointed",
    "disappointing",
    "waste",
    "problem",
    "issue"
]

In [54]:
keyword_counts = Counter()

for feedback in critical_reviews["Clean_Feedback"]:
    words = feedback.split()
    
    for word in words:
        if word in complaint_keywords:
            keyword_counts[word] += 1

In [55]:
keyword_counts.most_common(10)

[('bad', 12524),
 ('quality', 11396),
 ('waste', 9496),
 ('worst', 7504),
 ('poor', 6915),
 ('disappointed', 5536),
 ('problem', 1325),
 ('issue', 870),
 ('damaged', 801),
 ('broken', 581)]

In [56]:
top_complaints = keyword_counts.most_common(10)

for word, count in top_complaints:
    print(f"{word}: {count}")

bad: 12524
quality: 11396
waste: 9496
worst: 7504
poor: 6915
disappointed: 5536
problem: 1325
issue: 870
damaged: 801
broken: 581


In [57]:
critical_reviews["feedback_length"] = critical_reviews["Clean_Feedback"].str.len()

In [58]:
top_3_reviews = critical_reviews.sort_values(
    by="feedback_length",
    ascending=False
).head(3)

In [59]:
top_3_reviews[[
    "Product_name",
    "Rate",
    "Review",
    "Summary"
]]

,Product_name,Rate,Review,Summary
169765,"IFB 23 L Convection Microwave Oven??(23SC3, ...",1,Hard time following up with Flipkart for the i...,This time flipkart has disappointed me big tim...
76080,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...
78554,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...


In [60]:
for index, row in top_3_reviews.iterrows():
    print("=" * 80)
    print("Product:", row["Product_name"])
    print("Rating:", row["Rate"])
    print("Review:", row["Review"])
    print("Summary:", row["Summary"])

Product: IFB 23 L Convection Microwave Oven??(23SC3, Silver)
Rating: 1
Review: Hard time following up with Flipkart for the installation and demo of the IFB Microwave
Summary: This time flipkart has disappointed me big time, by not arranging the demo and installation till date. The IFB microwave oven was delivered to me on 29th Apr 2015 and it has been a month that I am still waiting for the installation and demo.The product is lying for one month now without use. Every time I call seeking an update from the customer care, I always get to hear to wait for 24 hrs and these 24 hrs have been never ending.Flipkart shoud not sale the product or else ensure proper se...
Product: BAJAJ Majesty New RCX 3 Electric Rice Cooker??(1.5 L, White and Red)
Rating: 1
Review: Necessity of Card Swipe Machine while Delivery of product
Summary: Don't Just wait for customers to give you ratings. The Product might be good or Bad. Today the person who delivered the product did not had the card swipe machi

In [61]:
selected_reviews = top_3_reviews[
    ["Product_name", "Rate", "Review", "Summary", "Clean_Feedback"]
].copy()

selected_reviews

,Product_name,Rate,Review,Summary,Clean_Feedback
169765,"IFB 23 L Convection Microwave Oven??(23SC3, ...",1,Hard time following up with Flipkart for the i...,This time flipkart has disappointed me big tim...,hard time following up with flipkart for the i...
76080,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...,necessity of card swipe machine while delivery...
78554,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...,necessity of card swipe machine while delivery...


In [62]:
selected_reviews = selected_reviews.reset_index(drop=True)

selected_reviews["Case"] = [
    "Case 1",
    "Case 2",
    "Case 3"
]

selected_reviews[[
    "Case",
    "Product_name",
    "Rate",
    "Review",
    "Summary"
]]

,Case,Product_name,Rate,Review,Summary
0,Case 1,"IFB 23 L Convection Microwave Oven??(23SC3, ...",1,Hard time following up with Flipkart for the i...,This time flipkart has disappointed me big tim...
1,Case 2,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...
2,Case 3,BAJAJ Majesty New RCX 3 Electric Rice Cooker?...,1,Necessity of Card Swipe Machine while Delivery...,Don't Just wait for customers to give you rati...


In [63]:
different_complaint_reviews = critical_reviews[
    critical_reviews["Clean_Feedback"].str.contains(
        "broken|damaged|defective|refund|poor quality|not working",
        case=False,
        na=False
    )
].copy()

different_complaint_reviews[
    ["Product_name", "Rate", "Review", "Summary"]
].head(10)

,Product_name,Rate,Review,Summary
12,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Don't waste your money,"Water pump is not throwing a lot of water, tha..."
20,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Utterly Disappointed,Never purchase cooler from FlipkartMy 10500 wa...
155,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Not recommended at all,this cooler is wonderful I would say best dese...
161,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Worthless,I did not like this product. Its fan doesn't w...
174,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Absolute rubbish!,Water pump was not working.
175,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Did not meet expectations,"Received defective product, auto swing is not ..."
187,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Hated it!,Very bad quality very poor plastic material ve...
199,"Crompton 75 L Desert Air Cooler??(White, Tea...",2,Damaged,Got a damaged one
217,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Really Nice,Product is good.. slightly noisy and too much ...
229,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Worst experience ever!,One wheel is damaged and so many cratches and ...


In [65]:
def create_prompt(review):
    prompt = f"""
You are a professional customer support agent.

Read the following customer complaint and write a short,
personalized and empathetic apology email.

Requirements:
- Address the specific complaint mentioned by the customer.
- Acknowledge the customer's frustration.
- Apologize sincerely.
- Keep the email professional and concise.
- Do not invent refunds, compensation, or actions that were not mentioned.
- Include a suitable subject line.

Customer complaint:
{review}
"""
    return prompt

In [66]:
test_prompt = create_prompt(selected_reviews.loc[0, "Clean_Feedback"])

print(test_prompt)


You are a professional customer support agent.

Read the following customer complaint and write a short,
personalized and empathetic apology email.

Requirements:
- Address the specific complaint mentioned by the customer.
- Acknowledge the customer's frustration.
- Apologize sincerely.
- Keep the email professional and concise.
- Do not invent refunds, compensation, or actions that were not mentioned.
- Include a suitable subject line.

Customer complaint:
hard time following up with flipkart for the installation and demo of the ifb microwave this time flipkart has disappointed me big time by not arranging the demo and installation till date the ifb microwave oven was delivered to me on 29th apr 2015 and it has been a month that i am still waiting for the installation and demothe product is lying for one month now without use every time i call seeking an update from the customer care i always get to hear to wait for 24 hrs and these 24 hrs have been never endingflipkart shoud not sal

In [67]:
final_reviews = different_complaint_reviews.loc[[12, 175, 229]].copy()

final_reviews[
    ["Product_name", "Rate", "Review", "Summary"]
]

,Product_name,Rate,Review,Summary
12,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Don't waste your money,"Water pump is not throwing a lot of water, tha..."
175,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Did not meet expectations,"Received defective product, auto swing is not ..."
229,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Worst experience ever!,One wheel is damaged and so many cratches and ...


In [68]:
final_reviews = final_reviews.reset_index(drop=True)

final_reviews["Case"] = [
    "Case 1",
    "Case 2",
    "Case 3"
]

final_reviews[
    ["Case", "Product_name", "Rate", "Review", "Summary"]
]

,Case,Product_name,Rate,Review,Summary
0,Case 1,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Don't waste your money,"Water pump is not throwing a lot of water, tha..."
1,Case 2,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Did not meet expectations,"Received defective product, auto swing is not ..."
2,Case 3,"Crompton 75 L Desert Air Cooler??(White, Tea...",1,Worst experience ever!,One wheel is damaged and so many cratches and ...


In [69]:
def create_prompt(review):
    prompt = f"""
You are a professional customer support agent.

Read the customer's complaint below and write a short,
personalized and empathetic apology email.

Requirements:
- Address the specific problem mentioned by the customer.
- Acknowledge the customer's frustration.
- Apologize sincerely.
- Keep the email professional and concise.
- Do not invent refunds, compensation, or promises.
- Include a suitable subject line.
- Do not mention that you are an AI.

Customer complaint:
{review}
"""
    return prompt

In [70]:
test_prompt = create_prompt(final_reviews.loc[0, "Clean_Feedback"])

print(test_prompt)


You are a professional customer support agent.

Read the customer's complaint below and write a short,
personalized and empathetic apology email.

Requirements:
- Address the specific problem mentioned by the customer.
- Acknowledge the customer's frustration.
- Apologize sincerely.
- Keep the email professional and concise.
- Do not invent refunds, compensation, or promises.
- Include a suitable subject line.
- Do not mention that you are an AI.

Customer complaint:
dont waste your money water pump is not throwing a lot of water thats why its throwing warm air not satisfied with this kind of pump quality i have paid a lot for this not worth the money there is no work of ice chamber i need a partial refund



In [71]:
!pip install -U google-genai

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ----------------------------------- ---- 1.0/1.2 MB 8.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.0/1.2 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 1.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----------------------------------- ---- 1.8/2.0 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 9.6 MB/s  0:00:00

   ---------------------------------------- 0/5 [websockets]
   ---------------------------------------- 0/5 [websockets]
  Attempting uninstall: pydantic-core
   ---------------------------------------- 0/5 [websockets]
    Found existing installation: pydantic_core 2.41.5
   ---------------------------------------- 0/5 [websockets]
   -------- ------------------------------- 1/5 [pydantic-core]
    Uninstalling pydantic_core-2.41.5:
   -------- ------------------------------- 1/5

In [73]:
import os

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6KjDch4odVQde6oYs3fjFvsD6wgmnwyOX_56lXdAewxDg"

In [74]:
from google import genai

client = genai.Client()

In [77]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain customer service in one simple sentence."
)

print(response.text)

Customer service is helping people solve problems and feel valued so they keep trusting the business.


In [78]:
def generate_email(review):

    prompt = f"""
You are a professional customer support agent.

Read the customer's complaint below and write a short,
personalized and empathetic apology email.

Requirements:
- Address the specific problem mentioned by the customer.
- Acknowledge the customer's frustration.
- Apologize sincerely.
- Keep the email professional and concise.
- Do not invent refunds, compensation, or promises.
- Include a suitable subject line.
- Do not mention that you are an AI.

Customer complaint:
{review}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [79]:
final_reviews["AI_Email"] = final_reviews["Clean_Feedback"].apply(generate_email)

final_reviews[["Case", "Product_name", "AI_Email"]]

,Case,Product_name,AI_Email
0,Case 1,"Crompton 75 L Desert Air Cooler??(White, Tea...",**Subject:** Regarding your recent water pump ...
1,Case 2,"Crompton 75 L Desert Air Cooler??(White, Tea...",**Subject:** Regarding your recent order and d...
2,Case 3,"Crompton 75 L Desert Air Cooler??(White, Tea...",Subject: Sincere apologies regarding the condi...


In [80]:
for _, row in final_reviews.iterrows():
    print("=" * 80)
    print(row["Case"])
    print("Product:", row["Product_name"])
    print("Rating:", row["Rate"])
    print("\nAI Generated Email:\n")
    print(row["AI_Email"])
    print("=" * 80)
    print()

Case 1
Product: Crompton 75 L Desert Air Cooler??(White, Teal, ACGC-DAC751)
Rating: 1

AI Generated Email:

**Subject:** Regarding your recent water pump purchase – We are here to help

Dear Customer,

Thank you for reaching out to us. I am very sorry to hear that your new water pump is not performing correctly and is blowing warm air, and I completely understand your frustration—especially given the high cost and your disappointment with the quality and the ice chamber. 

We take the quality of our products very seriously, and this is certainly not the experience we want our customers to have. I would like to look into this matter for you right away and explore the available options to help resolve this issue. 

Could you please reply to this email with your order number? Once I have that information, I will review your case immediately and guide you through our next steps. 

Thank you for your patience, and I look forward to helping you resolve this.

Sincerely,

Customer Support T

## Key Insights

### Critical Review Analysis

Reviews with a rating of 1 or 2 stars were classified as critical reviews using a rule-based approach.

### Top Complaint Keywords

The most frequently occurring complaint keywords in critical reviews were:

1. Bad
2. Quality
3. Waste
4. Worst
5. Poor
6. Disappointed
7. Problem
8. Issue
9. Damaged
10. Broken

These keywords indicate that product quality, damaged products, and poor customer experiences are common complaint areas.

### Generative AI Outreach

Three detailed critical reviews were selected from the dataset. The reviews represented different complaint types:

- Case 1: Water pump / water flow problem
- Case 2: Defective product / auto-swing problem
- Case 3: Damaged product / scratches

Gemini was used to generate short, personalized and empathetic customer-support emails for each case.